# Part 1 — Source data inspection and quality review

This notebook inspects all six sheets in `FSB Bank Transaction.xlsx`, documents their columns and relationships, and checks for damaged or inconsistent data. Run cells from top to bottom. The source-table ERD is shown below.

In [1]:
# Import dependencies for analytics
import pandas as pd 

## 1. Setup

Import pandas and display all DataFrame columns.

In [2]:
# Setting to display all columns in the DataFrame
pd.set_option('display.max_columns', None)

In [3]:
# Process of extracting data from xlsx
def extract_data_from_xlsx(file_path):
    data = pd.read_excel(file_path)
    return(data)

## 2. Read the transaction sheet

The existing extraction function reads the first sheet of the workbook. Preview five records to confirm the transaction layout.

In [4]:
transaction_filepath = './FSB Bank Transaction.xlsx'
data_extracted = extract_data_from_xlsx(file_path = transaction_filepath)    
data_extracted.head()

,transaction_id,transaction_reference,account_number,debit_or_credit,transaction_code,transaction_date,transaction_time,amount,transaction_remark,channel
0,TRX21325483579612,100001020252132548357,90000260801008087,C,TRX001,2025213,112726,5200000,SETORTUNAI VIA CASH DEPOSIT MACHINE 00047,ATM_CDM
1,TRX21335043336597,100001020252133504333,90000260801002198,C,TRX001,2025213,142301,4000000,SETORTUNAI VIA AGENT 00070,AGENT
2,TRX21343446066904,100001020252134344606,90000260801006643,C,TRX001,2025213,111501,7600000,SETORTUNAI VIA CASH RECYCLING MACHINE 00037,ATM_CRM
3,TRX21357196529986,100001020252135719652,90000260801008644,C,TRX001,2025213,105331,3100000,SETORTUNAI VIA AGENT 00183,AGENT
4,TRX21439859637972,100001020252143985963,90000260802006117,C,TRX001,2025214,121626,8400000,SETORTUNAI VIA CASH DEPOSIT MACHINE 00070,ATM_CDM


## 3. Profile transactions

The first output reports shape, memory, exact duplicate rows, data types, missing values, and distinct values. The next output shows numeric summaries, date ranges for datetime columns, and common text values. `transaction_date` and `transaction_time` are stored as integers, so their numeric summaries are not calendar statistics.

In [5]:
# Lightweight profiling with pandas (no HTML report or profiling package)
print(f"Rows: {len(data_extracted):,} | Columns: {data_extracted.shape[1]:,}")
print(f"Memory: {data_extracted.memory_usage(deep=True).sum() / 1024**2:.2f} MiB")
print(f"Duplicate rows: {data_extracted.duplicated().sum():,}")

column_profile = pd.DataFrame({
    "dtype": data_extracted.dtypes.astype(str),
    "non_null": data_extracted.count(),
    "missing": data_extracted.isna().sum(),
    "missing_pct": data_extracted.isna().mean().mul(100).round(2),
    "unique": data_extracted.nunique(dropna=True),
})
display(column_profile)


Rows: 1,640 | Columns: 10
Memory: 0.72 MiB
Duplicate rows: 0


,dtype,non_null,missing,missing_pct,unique
transaction_id,object,1640,0,0.0,1640
transaction_reference,object,1640,0,0.0,1329
account_number,int64,1640,0,0.0,122
debit_or_credit,object,1640,0,0.0,2
transaction_code,object,1640,0,0.0,5
transaction_date,int64,1640,0,0.0,51
transaction_time,int64,1640,0,0.0,1202
amount,int64,1640,0,0.0,913
transaction_remark,object,1640,0,0.0,1068
channel,object,1640,0,0.0,8


In [6]:
# Distribution and range checks for each column
numeric_columns = data_extracted.select_dtypes(include="number")
if not numeric_columns.empty:
    display(numeric_columns.describe().T)

datetime_columns = data_extracted.select_dtypes(include=["datetime", "datetimetz"])
if not datetime_columns.empty:
    display(datetime_columns.agg(["min", "max"]).T)

text_columns = data_extracted.select_dtypes(include=["object", "category", "string"])
for column in text_columns.columns:
    print(f"\n{column} — top values:")
    display(text_columns[column].value_counts(dropna=False).head(5).rename("count").to_frame())


transaction_id — top values:

transaction_reference — top values:

debit_or_credit — top values:

transaction_code — top values:

transaction_remark — top values:

channel — top values:


,count,mean,std,min,25%,50%,75%,max
account_number,1640.0,9.000026e+16,8.978964e+06,9.000026e+16,9.000026e+16,9.000026e+16,9.000026e+16,9.000026e+16
transaction_date,1640.0,2.025237e+06,1.146969e+01,2.025213e+06,2.025228e+06,2.025236e+06,2.025245e+06,2.025267e+06
transaction_time,1640.0,1.275498e+05,6.682317e+04,1.000000e+00,8.391550e+04,1.311800e+05,1.829345e+05,2.358240e+05
amount,1640.0,9.447713e+05,2.444623e+06,2.320000e+02,7.500000e+03,1.000000e+05,6.368105e+05,1.490000e+07


,count
transaction_id,
TRX21325483579612,1
TRX24237250248740,1
TRX24251255588874,1
TRX24250483519477,1
TRX24249785476572,1


,count
transaction_reference,
200002020252663151088,2
200002020252444528474,2
300003020252445017724,2
200002020252311796828,2
200002020252311383983,2


,count
debit_or_credit,
D,1419
C,221


,count
transaction_code,
TRX002,1246
TRX001,141
TRX091,137
TRX003,61
TRX092,55


,count
transaction_remark,
MONTHLY FEE SEP 2025,96
MONTHLY FEE AUG 2025,41
TRF_032_6587603669_Nadia Safitri,10
TRF_123_7556464170_Sri Wahyuni,10
TRF_095_4586850142_Yuda Pratama,8


,count
channel,
DGBANKING,850
SYSTEM,438
ATM,211
ATM_CDM,32
AGENT,32


## 4. Source tables and ERD

![ERD of the six source tables](part_one_ERD.png)

The solid relationships are `transaction.account_number → master_account.account_number`, `transaction.transaction_code → master_transaction_code.transaction_code`, `master_account.cif_number → master_customer.cif_number`, and `master_customer.branch_code → master_branch.branch_code`. `master_bank_code` is a lookup for bank codes that must be parsed from transaction remarks; no source column links to it directly. These are logical relationships to validate, not enforced database foreign keys.

In [7]:
# Read every worksheet into a dictionary of DataFrames
sheets = pd.read_excel(transaction_filepath, sheet_name=None)

sheet_summary = pd.DataFrame([
    {"sheet": name, "rows": len(df), "columns": len(df.columns),
     "missing_cells": int(df.isna().sum().sum()),
     "duplicate_rows": int(df.duplicated().sum())}
    for name, df in sheets.items()
])
display(sheet_summary)

,sheet,rows,columns,missing_cells,duplicate_rows
0,transaction,1640,10,0,0
1,master_account,123,7,0,0
2,master_customer,123,13,0,0
3,master_transaction_code,6,2,0,0
4,master_bank_code,38,3,0,0
5,master_branch,74,3,0,0


## 5. Column inventory and data dictionary

The following table shows the actual pandas type and missing count for every source column. The definitions below describe the intended business meaning; identifier and code fields should be treated as text even when Excel causes pandas to read them as integers.

| Sheet | Columns and meaning |
|---|---|
| `transaction` | `transaction_id`: unique transaction row; `transaction_reference`: logical transaction grouping, which can repeat; `account_number`: customer account; `debit_or_credit`: D for outgoing, C for incoming; `transaction_code`: type of transaction; `transaction_date`: Julian date `YYYYDDD`; `transaction_time`: integer `HHMMSS`; `amount`: transaction amount; `transaction_remark`: transaction details; `channel`: transaction channel. |
| `master_account` | `account_number`: account identifier; `cif_number`: customer identifier; `account_open_date`: account opening date; `account_currency_type`: currency; `account_status`: status; `product_code`: account product code; `product_name`: account product name. |
| `master_customer` | `cif_number`: customer identifier; `customer_name`: name; `gender`: gender code; `cif_open_date`: customer record opening date; `branch_code`: servicing branch; `customer_birth_place`: birthplace; `customer_birth_date`: birth date; `customer_address`: address; `customer_city`: city; `customer_phone`: phone; `occupation`: occupation; `married_status`: marital status; `customer_email`: email. |
| `master_transaction_code` | `transaction_code`: transaction type code; `transaction_desc`: type description. |
| `master_bank_code` | `bank_code`: bank identifier; `bank_swift_code`: SWIFT code; `bank_name`: bank name. |
| `master_branch` | `branch_code`: branch identifier; `branch_name`: branch name; `branch_location`: location. |

The transaction field meanings and date/time formats come from the exam brief. The master-field descriptions follow their column names and observed values.

In [8]:
# Actual type, completeness, and distinct-value count for every column
column_rows = []
for name, df in sheets.items():
    for column in df.columns:
        column_rows.append({
            "sheet": name, "column": column,
            "pandas_type": str(df[column].dtype),
            "missing": int(df[column].isna().sum()),
            "unique": int(df[column].nunique(dropna=True))
        })
display(pd.DataFrame(column_rows))

,sheet,column,pandas_type,missing,unique
0,transaction,transaction_id,object,0,1640
1,transaction,transaction_reference,object,0,1329
2,transaction,account_number,int64,0,122
3,transaction,debit_or_credit,object,0,2
4,transaction,transaction_code,object,0,5
5,transaction,transaction_date,int64,0,51
6,transaction,transaction_time,int64,0,1202
7,transaction,amount,int64,0,913
8,transaction,transaction_remark,object,0,1068
9,transaction,channel,object,0,8


## 6. Data quality checks

A repeated `transaction_reference` is not automatically an error: the brief allows multiple rows for transfers and fee-bearing transactions. Primary identifiers should be unique. The relationship checks count transaction rows whose referenced values are absent from a master sheet.

In [9]:
# Check uniqueness and relationships across the six sheets
keys = {
    "transaction": "transaction_id",
    "master_account": "account_number",
    "master_customer": "cif_number",
    "master_transaction_code": "transaction_code",
    "master_bank_code": "bank_code",
    "master_branch": "branch_code",
}
key_checks = pd.DataFrame([
    {"sheet": name, "key": key,
     "missing_keys": int(sheets[name][key].isna().sum()),
     "duplicate_keys": int(sheets[name][key].duplicated().sum())}
    for name, key in keys.items()
])
display(key_checks)

relationships = [
    ("transaction", "account_number", "master_account", "account_number"),
    ("transaction", "transaction_code", "master_transaction_code", "transaction_code"),
    ("master_account", "cif_number", "master_customer", "cif_number"),
    ("master_customer", "branch_code", "master_branch", "branch_code"),
]
relationship_checks = pd.DataFrame([
    {"source": f"{src}.{src_col}", "master": f"{dst}.{dst_col}",
     "unmatched_rows": int((~sheets[src][src_col].isin(sheets[dst][dst_col])).sum())}
    for src, src_col, dst, dst_col in relationships
])
display(relationship_checks)

,sheet,key,missing_keys,duplicate_keys
0,transaction,transaction_id,0,0
1,master_account,account_number,0,0
2,master_customer,cif_number,0,0
3,master_transaction_code,transaction_code,0,0
4,master_bank_code,bank_code,0,1
5,master_branch,branch_code,0,0


,source,master,unmatched_rows
0,transaction.account_number,master_account.account_number,16
1,transaction.transaction_code,master_transaction_code.transaction_code,0
2,master_account.cif_number,master_customer.cif_number,0
3,master_customer.branch_code,master_branch.branch_code,0


In [10]:
# Check date order and inspect records behind the main exceptions
accounts = sheets["master_account"]
customers = sheets["master_customer"]
customer_dates = customers.set_index("cif_number")["cif_open_date"]
late_customer_open = accounts["cif_number"].map(customer_dates) > accounts["account_open_date"]
print("Customer opening date later than account opening date:", int(late_customer_open.sum()))
print("Repeated transaction references:", int(sheets["transaction"]["transaction_reference"].duplicated().sum()))

print("\nTransactions with an account absent from master_account:")
missing_account = ~sheets["transaction"]["account_number"].isin(accounts["account_number"])
display(sheets["transaction"].loc[missing_account, "account_number"].value_counts().rename("rows").to_frame())

print("\nMaster bank rows sharing a bank code:")
banks = sheets["master_bank_code"]
display(banks.loc[banks["bank_code"].duplicated(keep=False)])

Customer opening date later than account opening date: 123
Repeated transaction references: 311

Transactions with an account absent from master_account:

Master bank rows sharing a bank code:


,rows
account_number,
90000260826004896,16


,bank_code,bank_swift_code,bank_name
19,116,PDJBIDJA,Bank Jatim
35,116,BPDKIDJA,Bank Aceh Syariah


## 7. Findings and next steps

- All six sheets have zero missing cells and zero exact duplicate rows in this workbook. These checks describe the source as read by pandas; they do not prove every value is valid.
- All source keys are unique except `master_bank_code.bank_code`: code `116` appears for **Bank Jatim** and **Bank Aceh Syariah**. A join on bank code alone could return two bank names; this needs clarification before building the datamart.
- **16 transaction rows** use account `90000260826004896`, which is absent from `master_account`. Those rows cannot receive customer or account details through the expected join. Confirm or restore the missing master record.
- **123 of 123 accounts** have a linked `cif_open_date` later than `account_open_date` (customer dates are in August 2026; account dates are in August 2025). Check the customer dates with the source owner before using account-aging logic.
- **311 repeated transaction references** are expected to be possible according to the exam brief. `transaction_id` remains unique. Review transaction types before treating any repeated reference as an error.
- `transaction_date` is `YYYYDDD` and `transaction_time` is an integer `HHMMSS`; convert them carefully in Part 2. Keep account and bank identifiers as text for joins and leading-zero preservation.